In [16]:
import pandas as pd

data1 = {
    "Nama": ["Ali", "Edi", "Annie", "Budiman", "Herman", "Didi", "Rina", "Gatot"],
    "Usia": ["muda", "muda", "muda", "tua", "tua", "muda", "tua", "tua"],
    "Berat": ["overweight", "underweight", "average", "overweight", "overweight", "underweight", "overweight", "average"],
    "Kelamin": ["pria", "pria", "wanita", "pria", "pria", "pria", "wanita", "pria"],
    "Hipertensi": ["ya", "tidak", "tidak", "tidak", "ya", "tidak", "ya", "tidak"]
}

df1 = pd.DataFrame(data1)

data2 = {
    "Day": ["D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14"],
    "Outlook": ["Sunny","Sunny","Overcast","Rain","Rain","Rain","Overcast","Sunny","Sunny","Rain","Sunny","Overcast","Overcast","Rain"],
    "Temperature": ["Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"],
    "Humidity": ["High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"],
    "Wind": ["Weak","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Strong"],
    "PlayTennis": ["No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"]
}

df2 = pd.DataFrame(data2)


In [17]:
import pandas as pd
import numpy as np
from collections import Counter

class DecisionTreeModel():
    def __init__(self):
        self.tree = None

    def entropy(self, y):
        counts = Counter(y)
        total = len(y)
        entropy = 0

        for count in counts.values():
            p = count / total
            entropy -= p * np.log2(p)

        return entropy

    def information_gain(self, X, y, feature):
        total_entropy = self.entropy(y)

        values = X[feature].unique()
        weighted_entropy = 0

        for value in values:
            subset_y = y[X[feature] == value]
            weighted_entropy += (len(subset_y) / len(y)) * self.entropy(subset_y)

        return total_entropy - weighted_entropy

    def build_tree(self, X, y):
        if len(set(y)) == 1:
            return y.iloc[0]

        if len(X.columns) == 0:
            return Counter(y).most_common(1)[0][0]

        gains = {}
        for feature in X.columns:
            gains[feature] = self.information_gain(X, y, feature)

        best_feature = max(gains, key=gains.get)

        tree = {best_feature: {}}

        for value in X[best_feature].unique():
            subset_X = X[X[best_feature] == value].drop(columns=[best_feature])
            subset_y = y[X[best_feature] == value]

            tree[best_feature][value] = self.build_tree(subset_X, subset_y)

        return tree

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        feature = list(tree.keys())[0]
        value = x[feature]

        if value in tree[feature]:
            return self.predict_one(x, tree[feature][value])
        else:
            return None

    def predict(self, X):
        return X.apply(lambda row: self.predict_one(row, self.tree), axis=1)

In [18]:
X = df1.drop("Hipertensi", axis=1)
y = df1["Hipertensi"]

model = DecisionTreeModel()
model.fit(X, y)

print("Tree:")
print(model.tree)

predictions = model.predict(X)
print("Predictions:")
print(predictions)
accuracy = (predictions == y).mean()
print("\nAccuracy:", accuracy)

Tree:
{'Nama': {'Ali': 'ya', 'Edi': 'tidak', 'Annie': 'tidak', 'Budiman': 'tidak', 'Herman': 'ya', 'Didi': 'tidak', 'Rina': 'ya', 'Gatot': 'tidak'}}
Predictions:
0       ya
1    tidak
2    tidak
3    tidak
4       ya
5    tidak
6       ya
7    tidak
dtype: str

Accuracy: 1.0


In [19]:
X = df2.drop("PlayTennis", axis=1)
y = df2["PlayTennis"]

model = DecisionTreeModel()
model.fit(X, y)

print("Tree:")
print(model.tree)

predictions = model.predict(X)

print("\nPredictions:")
print(predictions)

accuracy = (predictions == y).mean()
print("\nAccuracy:", accuracy)

Tree:
{'Day': {'D1': 'No', 'D2': 'No', 'D3': 'Yes', 'D4': 'Yes', 'D5': 'Yes', 'D6': 'No', 'D7': 'Yes', 'D8': 'No', 'D9': 'Yes', 'D10': 'Yes', 'D11': 'Yes', 'D12': 'Yes', 'D13': 'Yes', 'D14': 'No'}}

Predictions:
0      No
1      No
2     Yes
3     Yes
4     Yes
5      No
6     Yes
7      No
8     Yes
9     Yes
10    Yes
11    Yes
12    Yes
13     No
dtype: str

Accuracy: 1.0
